# LLM-as-a-Judge for Conversational Music Recommendation
This notebook runs the complete 2-stage pipeline:
1. **Stage 1 (Generation):** Open-source models generate music recommendation responses.
2. **Stage 2 (Evaluation):** Gemini evaluates responses across Personalization and Explanation Quality with multi-key rotation and pacing.

In [ ]:
# Step 0: Check GPU availability
!nvidia-smi

In [ ]:
# Step 1: Clone / Pull latest repository and install requirements
import os
repo_dir = "/content/llm-as-a-jude-for-crs"
if not os.path.exists(repo_dir):
    !git clone https://github.com/ibanana0/llm-as-a-jude-for-crs.git {repo_dir}
%cd {repo_dir}
!git pull
!pip install -r requirements.txt

In [ ]:
# Step 2: Set credentials (HF Token & Gemini API Keys)
import os
from google.colab import userdata
from huggingface_hub import login

# 1. Hugging Face Login
try:
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        login(token=hf_token)
    else:
        login()
except Exception:
    login()

# 2. Gemini API Keys (Supports multiple keys comma-separated for automatic rotation!)
gemini_keys = []
for key_var in ["GEMINI_API_KEYS", "GEMINI_API_KEY", "GEMINI_API_KEY_1", "GEMINI_API_KEY_2", "GEMINI_API_KEY_3"]:
    val = userdata.get(key_var)
    if val:
        gemini_keys.extend([k.strip() for k in val.split(",") if k.strip()])

if gemini_keys:
    clean_keys = list(dict.fromkeys(gemini_keys))
    os.environ["GEMINI_API_KEYS"] = ",".join(clean_keys)
    os.environ["GEMINI_API_KEY"] = clean_keys[0]
    print(f"Loaded {len(clean_keys)} Gemini API key(s) from Secrets for automatic rotation!")
else:
    # Fallback: Enter your keys separated by comma here if not using Colab Secrets
    raw_keys = "YOUR_KEY_1,YOUR_KEY_2"
    clean_keys = [k.strip() for k in raw_keys.split(",") if k.strip()]
    os.environ["GEMINI_API_KEYS"] = ",".join(clean_keys)
    os.environ["GEMINI_API_KEY"] = clean_keys[0] if clean_keys else ""
    print("Please set GEMINI_API_KEYS in Colab Secrets or paste them in raw_keys above.")


### [OPTION A] Restore Already Generated Responses from Google Drive
Run this cell if you **already generated responses** in a previous session and saved them to Google Drive.
This skips Stage 1 and immediately loads all 20 sessions and response files into `exp/`.

In [ ]:
# Restore from Google Drive
from google.colab import drive
import os

drive.mount("/content/drive")
drive_backup_dir = "/content/drive/MyDrive/llm_crs_experiment"

if os.path.exists(drive_backup_dir):
    %cd /content/llm-as-a-jude-for-crs
    !mkdir -p exp
    !cp -r "$drive_backup_dir"/* exp/
    print("✅ Restored existing responses & session IDs from Google Drive into exp/!")
    !ls -la exp
else:
    print(f"⚠️ No previous backup directory found at {drive_backup_dir}. Proceed to Option B to generate responses.")

### [OPTION B] Stage 1: Generate Responses with Open-Source Models
Run this if you need to generate responses for all 4 models (`Llama-1B`, `Llama-3B`, `Gemma-2B`, `Gemma-4B-4bit`).
*Note: If you restored from Google Drive in Option A, you can skip this step.*

In [ ]:
# Run batch inference across all models (auto-skips existing sessions, auto-enables 4-bit for Gemma-4B)
%cd /content/llm-as-a-jude-for-crs
!bash scripts/infer.sh

### Stage 2: Run LLM-as-a-Judge Evaluation (Gemini with Multi-Key Rotation)
Runs the full evaluation rubric with automatic key rotation and pacing (4s interval per key, 60s cooldown on 429/503).

In [ ]:
# Run the judge with automatic multi-key rotation
%cd /content/llm-as-a-jude-for-crs
!python eval_llm_as_a_judge.py \
  --judge_model gemini-3.8-flash

In [ ]:
# Summarize results (computes average scores & exports CSV and consolidated JSON)
%cd /content/llm-as-a-jude-for-crs
!python summarize_results.py --judge_model gemini-3.8-flash

### Stage 3: Backup Everything to Google Drive & View Summary

In [ ]:
# Backup exp directory to Google Drive
from google.colab import drive
import os

drive.mount("/content/drive")
drive_backup_dir = "/content/drive/MyDrive/llm_crs_experiment"
os.makedirs(drive_backup_dir, exist_ok=True)

!cp -r exp/* "$drive_backup_dir/"
print(f"✅ Successfully backed up all experiment artifacts to Google Drive:\n   {drive_backup_dir}")

In [ ]:
# Display summary dataframe
import pandas as pd

summary_df = pd.read_csv(f"{drive_backup_dir}/evaluation_summary.csv")
display(summary_df)